In [76]:
import numpy as np
from collections import Counter

def is_rack_stable(rack_weight, rack_height_mm, rack_cg_height_mm, servers, rack_depth_mm):
    """
    Checks if the rack configuration is likely stable based on the combined
    vertical center of gravity.
    """
    total_weight = rack_weight + sum(s[1] for s in servers)
    if total_weight == 0:
        return True

    combined_vertical_cg = (rack_weight * rack_cg_height_mm +
                             sum(s[1] * s[0] for s in servers)) / total_weight

    stability_threshold_fraction = 0.7  # Adjust as needed
    return combined_vertical_cg <= rack_height_mm * stability_threshold_fraction

def find_stable_positions_greedy_complex(rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
                                         servers_to_place, server_specs, prioritize_top=False):
    """
    A greedy approach to find stable server positions for various server types.
    """
    u_height_mm = 44.45
    rack_height_mm = rack_height_u * u_height_mm
    rack_cg_height_mm = rack_height_mm / 2

    all_servers = []
    for server_type, count in servers_to_place.items():
        specs = server_specs.get(server_type)
        if specs:
            for _ in range(count):
                all_servers.append({'type': server_type, 'height': specs['height'], 'weight': specs['weight']})
        else:
            print(f"Warning: Specifications not found for server type '{server_type}'. Skipping.")

    if not all_servers:
        return {}  # Return empty dict for empty Counter

    if not prioritize_top:
        placed_servers_info = []
        occupied_u = [False] * rack_height_u
        for server in all_servers:
            server_height_u = server['height']
            server_weight_kg = server['weight']
            server_cg_offset = (server_height_u * u_height_mm) / 2
            placed = False
            for i in range(rack_height_u):
                if not occupied_u[i]:
                    start_u = i
                    server_base_height = start_u * u_height_mm
                    server_cg = server_base_height + server_cg_offset
                    can_place = True
                    for u_check in range(start_u, start_u + server_height_u):
                        if u_check >= rack_height_u or occupied_u[u_check]:
                            can_place = False
                            break
                    if can_place:
                        temp_positions = [(p['cg'], p['weight']) for p in placed_servers_info] + [(server_cg, server_weight_kg)]
                        if is_rack_stable(rack_weight_kg, rack_height_mm, rack_cg_height_mm, temp_positions, rack_depth_mm):
                            placed_servers_info.append({'cg': server_cg, 'weight': server_weight_kg, 'type': server['type'], 'height': server['height'], 'start_u': start_u})
                            for u in range(start_u, start_u + server_height_u):
                                if u < rack_height_u:
                                    occupied_u[u] = True
                            placed = True
                            break
            if not placed:
                return None

        final_placement = {}
        for i, server_info in enumerate(placed_servers_info):
            final_placement[f"{server_info['type']}_{i+1}"] = server_info['start_u'] + 1
        return final_placement

    else:
        placed_servers_info = []
        occupied_u = [False] * rack_height_u
        for server in all_servers:
            server_height_u = server['height']
            server_weight_kg = server['weight']
            server_cg_offset = (server_height_u * u_height_mm) / 2
            placed = False
            for i in range(rack_height_u - 1, -1, -1):
                if not occupied_u[i]:
                    start_u = i
                    server_base_height = start_u * u_height_mm
                    server_cg = server_base_height + server_cg_offset
                    can_place = True
                    for u_check in range(start_u, start_u + server_height_u):
                        if u_check >= rack_height_u or occupied_u[u_check]:
                            can_place = False
                            break
                    if can_place:
                        temp_positions = [(p['cg'], p['weight']) for p in placed_servers_info] + [(server_cg, server_weight_kg)]
                        if is_rack_stable(rack_weight_kg, rack_height_mm, rack_cg_height_mm, temp_positions, rack_depth_mm):
                            placed_servers_info.append({'cg': server_cg, 'weight': server_weight_kg, 'type': server['type'], 'height': server['height'], 'start_u': start_u})
                            for u in range(start_u, start_u + server_height_u):
                                if u < rack_height_u:
                                    occupied_u[u] = True
                            placed = True
                            break
            if not placed:
                return None

        final_placement = {}
        sorted_servers = sorted(placed_servers_info, key=lambda x: x['start_u'], reverse=True)
        occupied_map = [False] * rack_height_u
        placed_index = 1
        for server in sorted_servers:
            start_u = server['start_u']
            server_type = server['type']
            server_height = server['height']
            for u in range(start_u, -1, -1):
                can_place_here = True
                for check_u in range(u, u + server_height):
                    if check_u >= rack_height_u or occupied_map[check_u]:
                        can_place_here = False
                        break
                if can_place_here:
                    final_placement[f"{server_type}_{placed_index}"] = u + 1
                    for occupy_u in range(u, u + server_height):
                        if occupy_u < rack_height_u:
                            occupied_map[occupy_u] = True
                    placed_index += 1
                    break
            else:
                # Fallback to the initially found stable position
                final_placement[f"{server_type}_{placed_index}"] = start_u + 1
                for occupy_u in range(start_u, start_u + server_height):
                    if occupy_u < rack_height_u:
                        occupied_map[occupy_u] = True
                placed_index += 1

        return final_placement

# --- Example Usage ---
rack_height_u = 42
rack_weight_kg = 114.55
rack_width_mm = 600
rack_depth_mm = 1200

all_racks_config = [
    Counter({'node_1': 5, 'node_2': 1, 'node_4': 1}),
    Counter({'node_4': 12, 'node_2': 1}),
    Counter({'node_1': 5, 'node_2': 1, 'node_4': 1}),
    Counter({'node_4': 15, 'node_1': 5}),
    Counter({'node_1': 5, 'node_2': 1, 'node_4': 1}),
    Counter({'node_3': 8}),
    Counter({'node_3': 15}),
    Counter({'node_3': 7, 'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter({'node_2': 1}),
    Counter(),
    Counter(),
]

colors_info = {
    'node_1': {'count': 20, 'wattage': 1600, 'height': 2, 'weight': 30, 'LAN_200GNDR': 1, 'LAN_25G': 2},
    'node_2': {'count': 20, 'wattage': 11000, 'height': 8, 'weight': 15, '200GNDR': 1, 'LAN_400GEth': 8, 'LAN_25G': 2},
    'node_3': {'count': 30, 'wattage': 1250, 'height': 2, 'weight': 20, 'LAN_25G': 2, 'LAN_200GNDR': 2},
    'node_4': {'count': 30, 'wattage': 750, 'height': 1, 'weight': 9, 'LAN_25G': 2}
}

results = []

for rack_config in all_racks_config:
    if rack_config:
        print(f"Processing rack with config: {rack_config}")
        # Bottom-up placement
        stable_placement_bottom = find_stable_positions_greedy_complex(
            rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
            rack_config, colors_info, prioritize_top=False
        )
        print(f"  Bottom-up Placement: {stable_placement_bottom}")

        # Top-biased placement
        stable_placement_top_biased = find_stable_positions_greedy_complex(
            rack_height_u, rack_weight_kg, rack_width_mm, rack_depth_mm,
            rack_config, colors_info, prioritize_top=True
        )
        print(f"  Top-biased Placement: {stable_placement_top_biased}")
        print("-" * 30)

print("Processing complete.")

Processing rack with config: Counter({'node_1': 5, 'node_2': 1, 'node_4': 1})
  Bottom-up Placement: {'node_1_1': 1, 'node_1_2': 3, 'node_1_3': 5, 'node_1_4': 7, 'node_1_5': 9, 'node_2_6': 11, 'node_4_7': 19}
  Top-biased Placement: {'node_1_1': 41, 'node_1_2': 39, 'node_1_3': 37, 'node_4_4': 36, 'node_1_5': 32, 'node_1_6': 30, 'node_2_7': 22}
------------------------------
Processing rack with config: Counter({'node_4': 12, 'node_2': 1})
  Bottom-up Placement: {'node_4_1': 1, 'node_4_2': 2, 'node_4_3': 3, 'node_4_4': 4, 'node_4_5': 5, 'node_4_6': 6, 'node_4_7': 7, 'node_4_8': 8, 'node_4_9': 9, 'node_4_10': 10, 'node_4_11': 11, 'node_4_12': 12, 'node_2_13': 13}
  Top-biased Placement: {'node_4_1': 42, 'node_4_2': 41, 'node_4_3': 40, 'node_4_4': 39, 'node_4_5': 38, 'node_4_6': 37, 'node_4_7': 36, 'node_4_8': 35, 'node_4_9': 34, 'node_4_10': 33, 'node_4_11': 32, 'node_4_12': 31, 'node_2_13': 23}
------------------------------
Processing rack with config: Counter({'node_1': 5, 'node_2': 1

In [74]:
import numpy as np
import os
import time
import multiprocessing as mp
from collections import Counter
from itertools import product, permutations
from math import ceil
import logging
import random

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def calculate_box_wattage(box, colors_wattage):
    return sum(box.get(color, 0) * colors_wattage.get(color, 0) for color in colors_wattage)

def calculate_box_height(box, colors_height):
    return sum(box.get(color, 0) * colors_height.get(color, 0) for color in colors_height)

def check_box_limits(box, colors_wattage, colors_height, max_box_wattage, max_box_height):
    box_wattage = calculate_box_wattage(box, colors_wattage)
    box_height = calculate_box_height(box, colors_height)
    return box_wattage <= max_box_wattage and box_height <= max_box_height

def display_distribution_filled_only(dist, colors_wattage, colors_height, total_balls, balls_placed_overall):
    print(f"\nValid Distribution Found (Iterative Greedy - Filled Racks Only):")
    current_total_balls = sum(balls_placed_overall.values())
    total_wattage = 0
    total_height = 0
    total_count = 0
    filled_boxes = []
    
    for i, box in enumerate(dist):
        if box:
            rack_config = box
            box_contents = ", ".join(f"{count} {color}" for color, count in box.items())
            box_wattage = calculate_box_wattage(box, colors_wattage)
            box_height = calculate_box_height(box, colors_height)
            total_wattage += box_wattage
            total_height += box_height
            ball_count = sum(box.values())
            total_count += ball_count
            filled_boxes.append(f"  Box {i+1}: {box_contents} (Wattage: {box_wattage}, Height: {box_height}, count:{ball_count})")
    if filled_boxes:
        for box_info in filled_boxes:
            print(box_info)
        print(f"  Total Wattage: {total_wattage}, Total Height: {total_height}, Total Balls Distributed:{total_count} {current_total_balls}")
    else:
        print(f"  No racks were filled. Total Balls Distributed: {current_total_balls}")
    if current_total_balls != total_balls:
        print(f"  WARNING: Total balls distributed ({current_total_balls}) does not match the expected total ({total_balls})!")

def find_valid_distributions_iterative_greedy_adaptive(colors_info, initial_num_boxes, max_box_wattage, max_box_height, total_balls):
    colors = list(colors_info.keys())
    colors_wattage = {color: colors_info[color]['wattage'] for color in colors}
    colors_height = {color: colors_info[color]['height'] for color in colors}
    max_possible_boxes = total_balls
    valid_distributions = []
    seen_distributions = set()

    def get_distribution_signature(distribution):
        filled_boxes_signature = tuple(sorted(tuple(sorted(box.items())) for box in distribution if box))
        return filled_boxes_signature

    num_boxes_to_try = list(range(initial_num_boxes, max_possible_boxes + 1))
    random.shuffle(num_boxes_to_try)

    for num_boxes in num_boxes_to_try:
        distributions = [Counter() for _ in range(num_boxes)]
        balls_placed = {color: 0 for color in colors}
        box_index = 0

        while box_index < num_boxes and any(balls_placed[color] < colors_info[color]['count'] for color in colors):
            current_box = distributions[box_index]
            remaining_balls = {c: colors_info[c]['count'] - balls_placed[c] for c in colors}
            possible_colors = [c for c in colors if remaining_balls[c] > 0]

            if not possible_colors:
                box_index += 1
                continue

            scenarios = []
            for color in possible_colors:
                scenarios.append(("single_color", color))
                scenarios.append(("partial_color_50", color))
                scenarios.append(("partial_color_25", color))
                scenarios.append(("partial_color_10", color))
                scenarios.append(("mixed_fill_initial", color))

            random.shuffle(scenarios) # Try scenarios in a random order for each box

            applied_scenario = False
            for scenario_type, main_color in scenarios:
                temp_box = current_box.copy()
                temp_balls_placed = balls_placed.copy()
                temp_remaining_balls = remaining_balls.copy()

                if scenario_type == "single_color":
                    while temp_remaining_balls.get(main_color, 0) > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                            calculate_box_wattage(temp_box, colors_wattage) + colors_wattage.get(main_color, 0) <= max_box_wattage and \
                            calculate_box_height(temp_box, colors_height) + colors_height.get(main_color, 0) <= max_box_height:
                        temp_box[main_color] = temp_box.get(main_color, 0) + 1
                        temp_balls_placed[main_color] = temp_balls_placed.get(main_color, 0) + 1
                        temp_remaining_balls[main_color] -= 1

                    # Iterate through other colors if there are remaining balls
                    #if any(count > 0 for color, count in temp_remaining_balls.items() if color != main_color):
                    for other_color in list(temp_remaining_balls.keys()):  # Iterate over a copy to allow modification
                        if other_color != main_color and temp_remaining_balls.get(other_color, 0) > 0:
                            while temp_remaining_balls[other_color] > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                    calculate_box_wattage(temp_box, colors_wattage) + colors_wattage.get(other_color, 0) <= max_box_wattage and \
                                    calculate_box_height(temp_box, colors_height) + colors_height.get(other_color, 0) <= max_box_height:
                                temp_box[other_color] = temp_box.get(other_color, 0) + 1
                                temp_balls_placed[other_color] = temp_balls_placed.get(other_color, 0) + 1
                                temp_remaining_balls[other_color] -= 1

                    if temp_box != current_box:
                        distributions[box_index] = temp_box
                        balls_placed.update(temp_balls_placed)
                        applied_scenario = True
                    # No need to break here, continue to the next color

                    if applied_scenario: # Only break if any color was added after the main color
                        break

                elif scenario_type.startswith("partial_color"):
                    percentage = float(int(scenario_type.split('_')[2])/100)
                    add_amount = min(ceil(colors_info[main_color]['count'] * percentage), temp_remaining_balls[main_color])
                    can_add = True
                    for _ in range(add_amount):
                        if check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                           calculate_box_wattage(temp_box, colors_wattage) + colors_wattage[main_color] <= max_box_wattage and \
                           calculate_box_height(temp_box, colors_height) + colors_height[main_color] <= max_box_height and \
                           temp_remaining_balls[main_color] > 0:
                            temp_box[main_color] += 1
                            temp_balls_placed[main_color] += 1
                            temp_remaining_balls[main_color] -= 1
                        else:
                            can_add = False
                            break
                    # Iterate through other colors if there are remaining balls
                    if any(count > 0 for color, count in temp_remaining_balls.items() if color != main_color):
                        for other_color in list(temp_remaining_balls.keys()):  # Iterate over a copy to allow modification
                            if other_color != main_color and temp_remaining_balls.get(other_color, 0) > 0:
                                while temp_remaining_balls[other_color] > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                                        calculate_box_wattage(temp_box, colors_wattage) + colors_wattage.get(other_color, 0) <= max_box_wattage and \
                                        calculate_box_height(temp_box, colors_height) + colors_height.get(other_color, 0) <= max_box_height:
                                    temp_box[other_color] = temp_box.get(other_color, 0) + 1
                                    temp_balls_placed[other_color] = temp_balls_placed.get(other_color, 0) + 1
                                    temp_remaining_balls[other_color] -= 1

                    if (can_add or add_amount > 0) and temp_box != current_box:
                        distributions[box_index] = temp_box
                        balls_placed.update(temp_balls_placed)
                        applied_scenario = True
                        break

                elif scenario_type == "mixed_fill_initial":
                    while temp_remaining_balls[main_color] > 0 and check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                          calculate_box_wattage(temp_box, colors_wattage) + colors_wattage[main_color] <= max_box_wattage and \
                          calculate_box_height(temp_box, colors_height) + colors_height[main_color] <= max_box_height:
                        temp_box[main_color] += 1
                        temp_balls_placed[main_color] += 1
                        temp_remaining_balls[main_color] -= 1

                    other_colors = [c for c in possible_colors if c != main_color]
                    random.shuffle(other_colors)
                    while check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                          calculate_box_wattage(temp_box, colors_wattage) < max_box_wattage and \
                          calculate_box_height(temp_box, colors_height) < max_box_height and \
                          any(temp_remaining_balls[c] > 0 for c in other_colors):
                        added_other = False
                        for other_color in other_colors:
                            if temp_remaining_balls[other_color] > 0 and \
                               check_box_limits(temp_box, colors_wattage, colors_height, max_box_wattage, max_box_height) and \
                               calculate_box_wattage(temp_box, colors_wattage) + colors_wattage[other_color] <= max_box_wattage and \
                               calculate_box_height(temp_box, colors_height) + colors_height[other_color] <= max_box_height:
                                temp_box[other_color] += 1
                                temp_balls_placed[other_color] += 1
                                temp_remaining_balls[other_color] -= 1
                                added_other = True
                                break
                        if not added_other:
                            break
                    if temp_box != current_box:
                        distributions[box_index] = temp_box
                        balls_placed.update(temp_balls_placed)
                        applied_scenario = True
                        break

            if applied_scenario or not possible_colors:
                box_index += 1
            elif not any(remaining_balls.values()):
                box_index += 1

        if sum(balls_placed.values()) == total_balls:
            signature = get_distribution_signature(distributions)
            if signature not in seen_distributions:
                seen_distributions.add(signature)
                valid_distributions.append(list(distributions))
                display_distribution_filled_only(distributions, colors_wattage, colors_height, total_balls, balls_placed)
                
                # Potentially break here if you only need one solution
                # break

    return valid_distributions



In [71]:
def main_iterative_greedy_adaptive_with_height():
    colors_info = {
        'node_1': {'count': 20, 'wattage': 1600, 'height': 2, 'weight':30, 'LAN_200GNDR':1, 'LAN_25G':2},
        'node_2': {'count': 20, 'wattage': 11000, 'height': 8, 'weight': 15, '200GNDR':1, 'LAN_400GEth':8, 'LAN_25G':2},
        'node_3': {'count': 30, 'wattage': 1250, 'height': 2, 'weight':20, 'LAN_25G':2, 'LAN_200GNDR':2},
        'node_4': {'count': 30, 'wattage': 750, 'height': 1, 'weight':9, 'LAN_25G':2}
    }
    num_devices = sum(info['count'] for info in colors_info.values())
    max_needed_wattage = sum(colors_info[color]['wattage'] * colors_info[color]['count'] for color in colors_info)
    max_box_wattage = 20000
    max_box_height = 42
    initial_num_boxes = ceil(max_needed_wattage / max_box_wattage) + 2

    print(f"Distributing {num_devices} balls into initially {initial_num_boxes} boxes which consumes {max_needed_wattage} watts (Iterative Greedy Adaptive with Height):")

    start_time = time.time()
    distributions = find_valid_distributions_iterative_greedy_adaptive(
        colors_info, initial_num_boxes, max_box_wattage, max_box_height, num_devices
    )
    end_time = time.time()

    print(f"\nSummary: Found {len(distributions)} unique distribution(s) in {end_time - start_time:.6f} seconds")

if __name__ == "__main__":
    main_iterative_greedy_adaptive_with_height()

Distributing 100 balls into initially 18 boxes which consumes 312000 watts (Iterative Greedy Adaptive with Height):

Valid Distribution Found (Iterative Greedy - Filled Racks Only):
boxxxxx [Counter({'node_1': 5, 'node_2': 1, 'node_4': 1}), Counter({'node_4': 12, 'node_2': 1}), Counter({'node_1': 5, 'node_2': 1, 'node_4': 1}), Counter({'node_4': 15, 'node_1': 5}), Counter({'node_1': 5, 'node_2': 1, 'node_4': 1}), Counter({'node_3': 8}), Counter({'node_3': 15}), Counter({'node_3': 7, 'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter({'node_2': 1}), Counter(), Counter(), Counter(), Counter(), Counter(), Counter(), Counter(), Counter(), Counter(), Counter(), Counter(), Counter(), Count